In [1]:
%pip install -qU langchain-community pymupdf
!pip install -qU langchain-huggingface sentence-transformers
!pip install -qU langchain-groq
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 26.6 MB/s eta 0:00:00


# 1. Loading the document

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/content/the-quran-with-annotated-interpretation-in-modern-english-ali-unal.pdf"
loader = PyMuPDFLoader(file_path)

In [3]:
docs = loader.load()
# skiping empty pages
non_empty_docs = [d for d in docs if d.page_content.strip()]

# 2. Spliting document into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000, # 10000 charecters long text
    chunk_overlap=200, # 200 charecters long overlapping
)

split_docs = text_splitter.split_documents(non_empty_docs)

# 3. Embeddings Model

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
texts = [doc.page_content for doc in split_docs] # convert documents into list[str]

In [7]:
split_docs_embeddings = embed_model.embed_documents(texts) # generate embeddings (list[list[float]])

# 4. FAISS (Facebook AI Similarity Search) vector database

In [8]:
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(
    documents=split_docs,
    embedding=embed_model,
)

# 5. LLM. GROQ (llama-3.3-70b-versatile)

In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="your_api_key"
)

# 6. Building Prompts and chains

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

In [15]:
def ask(query):
    prompt_template = PromptTemplate.from_template(
        "Give answer according to the following passages in the quran:\n{context}\n\n"
        "If the answer is not present in the given context then give answer according to "
        "the internet sources, but do inform that there are no passages in the quran "
        "about the question.\n\n"
        "Answer the following question:\n{question}"
    )

    def get_joined_context(q):
        docs = faiss_db.similarity_search(q, k=10)
        return "\n\n".join(d.page_content for d in docs)

    parallel_chain = RunnableParallel(
        context=RunnableLambda(get_joined_context),
        question=RunnablePassthrough()
    )

    chain = parallel_chain | prompt_template | llm | StrOutputParser()

    return chain.invoke(query)

Relevant questions

In [16]:
answer = ask("What does the Quran say about Day of Judgment?")
print(answer)

The Quran mentions the Day of Judgment in several passages, including the ones provided. According to the Quran, the Day of Judgment is a time when all souls will be held accountable for their deeds, and God's command will be absolute and exclusive (Surah 82:19). On this day, no person will be wronged in the least, and even the smallest deed will be weighed (Surah 21:47).

The Quran also describes the Day of Judgment as a time of great terror and punishment for the disbelieving criminals, who will be plunged into despair and punished in the Blazes (Surah 30:12-16, Surah 54:46-48). The righteous, on the other hand, will be honored and made happy in a delightful Garden (Surah 30:15).

The Quran emphasizes the importance of preparing for the Day of Judgment by living a righteous life, remembering God, and seeking forgiveness for one's sins (Surah 3:191-194). It also warns against associating partners with God and disobeying His commandments, which can lead to punishment on the Day of Judg

In [17]:
answer = ask("What is the importance of RAMADAN is islam?")
print(answer)

According to the passages provided from the Quran, Ramadan is a holy month in Islam where the Quran was sent down from Allah (Surah 44: 943, Surah 97: 1). The Night of Destiny (Laylat al-Qadr) occurs during this month, which is considered a night of great worth and significance (Surah 97: 1-5). Spending this night in devotion is highly valued, and it is believed that Allah's decrees for every affair are made during this night (Surah 97: 4).

In Surah 2: 185, it is mentioned that the Quran was sent down during the holy month of Ramadan. The Quran also emphasizes the importance of fasting during Ramadan, which is one of the Five Pillars of Islam (although not directly mentioned in the provided passages).

From external sources, we know that Ramadan is the ninth month of the Islamic calendar and is considered a sacred month for Muslims. During Ramadan, Muslims fast from dawn to sunset, abstaining from food and drink, to develop self-control, empathy for those less fortunate, and to streng

Irrelevant questions

In [18]:
answer = ask("When did dinosaurs came into being?")
print(answer)

There are no passages in the Quran about dinosaurs or their origin. However, according to internet sources and scientific research, dinosaurs are believed to have appeared on Earth during the Middle to Late Triassic period, around 230-245 million years ago. They dominated the Earth's landscapes during the Mesozoic Era, which lasted until about 65 million years ago, when they became extinct in an event known as the K-Pg extinction.

It's worth noting that the Quran does mention the creation of animals and the natural world, but it does not provide specific information about dinosaurs or their timeline. The Quran's focus is on the spiritual and moral guidance of humanity, rather than on scientific or historical details.
